# DenseNet121 512x512 production-ROI Grad-CAM experiment

This is the one remaining architecture-resolution test. DenseNet121 remains unchanged; only input resolution changes from 384 to 512. Training and validation use the exact fixed production YOLO crop: 1.15 expansion, square crop, black padding outside the source image. CE, sampler, and preprocessing remain unchanged.

A 512 input produces a denser final spatial feature grid than 384, which may make Grad-CAM less block-like. It does not guarantee more faithful evidence. Use the candidate only if fixed-production ROI metrics do not regress and its CAM audit improves.

In [ ]:
!pip -q install 'timm>=1.0' 'pandas>=2'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, math, random
from datetime import datetime, timezone
from pathlib import Path
import cv2, matplotlib.pyplot as plt, numpy as np, pandas as pd, timm, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


In [ ]:
SEED, INPUT_SIZE, BATCH_SIZE, NUM_WORKERS, EPOCHS = 42, 512, 24, 2, 5
LR, WEIGHT_DECAY, EXPANSION, CAM_PER_GRADE = 1e-5, 1e-3, 1.15, 25
ROOT = Path('/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1')
MANIFEST = ROOT / 'derived/production_yolo_roi_manifest_v1.csv'
BASE = Path('/content/drive/MyDrive/Models/densenet121_paired_view_adaptation/2026-07-30_09-03-29_850983_UTC/best_model.pth')
STAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = Path('/content/drive/MyDrive/Models/densenet121_production_roi_512_gradcam') / STAMP
for path in (MANIFEST, BASE):
    if not path.is_file(): raise FileNotFoundError(path)
RUN_DIR.mkdir(parents=True, exist_ok=False)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| output:', RUN_DIR)
frame = pd.read_csv(MANIFEST)
print(frame.groupby(['split','grade']).size().unstack(fill_value=0))


## Exact production ROI dataset

In [ ]:
def square_roi(image, box):
    h, w = image.shape[:2]; x1, y1, x2, y2 = map(float, box)
    side = int(math.ceil(max(x2-x1, y2-y1) * EXPANSION)); cx, cy = (x1+x2)/2, (y1+y2)/2
    ax, ay = int(math.floor(cx-side/2)), int(math.floor(cy-side/2)); bx, by = ax+side, ay+side
    crop = image[max(0,ay):min(h,by), max(0,ax):min(w,bx)]
    if crop.size == 0: raise RuntimeError(f'Empty ROI for {box}')
    return cv2.copyMakeBorder(crop, max(0,-ay), max(0,by-h), max(0,-ax), max(0,bx-w), cv2.BORDER_CONSTANT, value=(0,0,0))

class CLAHE:
    def __call__(self, image):
        lab=cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2LAB); l,a,b=cv2.split(lab)
        l=cv2.createCLAHE(clipLimit=1.25,tileGridSize=(8,8)).apply(l)
        return cv2.cvtColor(cv2.merge((l,a,b)),cv2.COLOR_LAB2RGB)

train_tf=transforms.Compose([CLAHE(),transforms.ToPILImage(),transforms.RandomHorizontalFlip(.5),transforms.RandomRotation(5),transforms.ColorJitter(brightness=.08,contrast=.08),transforms.Resize((INPUT_SIZE,INPUT_SIZE)),transforms.ToTensor(),transforms.RandomErasing(.10,scale=(.02,.05),ratio=(.5,2),value=0),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf=transforms.Compose([CLAHE(),transforms.ToPILImage(),transforms.Resize((INPUT_SIZE,INPUT_SIZE)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

class ROIs(Dataset):
    def __init__(self, data, transform): self.data=data.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        row=self.data.iloc[i]; image=cv2.imread(row.full_image,cv2.IMREAD_COLOR)
        if image is None: raise RuntimeError(row.full_image)
        roi=square_roi(image,[row.x1,row.y1,row.x2,row.y2])
        return self.transform(cv2.cvtColor(roi,cv2.COLOR_BGR2RGB)),int(row.grade),str(row.full_image)

class Model(nn.Module):
    def __init__(self): super().__init__(); self.backbone=timm.create_model('densenet121',pretrained=False,num_classes=5,drop_rate=.20)
    @property
    def gradcam_target_layer(self): return self.backbone.features.norm5
    def forward(self,x): return self.backbone(x)


## Baseline, fine-tune, and fixed-production validation

In [ ]:
def metrics(model, loader):
    labels=[]; probabilities=[]; model.eval()
    with torch.inference_mode():
        for images,y,_ in loader:
            probabilities.extend(F.softmax(model(images.to(DEVICE)).float(),1).cpu().numpy()); labels.extend(y.numpy())
    labels=np.asarray(labels); probabilities=np.asarray(probabilities); prediction=probabilities.argmax(1)
    p,r,f,_=precision_recall_fscore_support(labels,prediction,labels=range(5),average='macro',zero_division=0)
    return {'accuracy':float((labels==prediction).mean()),'qwk':float(cohen_kappa_score(labels,prediction,weights='quadratic')),'macro_precision':float(p),'macro_recall':float(r),'macro_f1':float(f),'macro_ap':float(average_precision_score(np.eye(5)[labels],probabilities,average='macro'))}

train_frame=frame[frame['split']=='train']; val_frame=frame[frame['split']=='val']
counts=np.bincount(train_frame.grade.to_numpy(),minlength=5); sample_weight=(1/counts)[train_frame.grade.to_numpy()]
train_loader=DataLoader(ROIs(train_frame,train_tf),batch_size=BATCH_SIZE,sampler=WeightedRandomSampler(torch.as_tensor(sample_weight,dtype=torch.double),len(sample_weight),replacement=True),num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(ROIs(val_frame,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
payload=torch.load(BASE,map_location='cpu',weights_only=False); model=Model().to(DEVICE); model.load_state_dict(payload['model_state_dict'],strict=True)
baseline=metrics(model,val_loader); print('Baseline at 512:',json.dumps(baseline,indent=2))
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=1e-7); scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
history=[]; best=-float('inf')
for epoch in range(1,EPOCHS+1):
    model.train(); total=0.; n=0
    for images,y,_ in tqdm(train_loader,desc=f'epoch {epoch}/{EPOCHS}'):
        images,y=images.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=DEVICE.type=='cuda'): loss=F.cross_entropy(model(images),y)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); scaler.step(optimizer); scaler.update(); total+=loss.item()*len(y); n+=len(y)
    scheduler.step(); result=metrics(model,val_loader); score=.55*result['qwk']+.30*result['macro_f1']+.15*result['macro_ap']; row={'epoch':epoch,'train_loss':total/n,'selection_score':score,**result}; history.append(row); print(json.dumps(row,indent=2))
    if score>best:
        best=score; torch.save({'model_state_dict':model.state_dict(),'architecture':'timm_densenet121_linear_gradcam','loss_type':'ce','epoch':epoch,'input_size':INPUT_SIZE,'base_checkpoint':str(BASE),'roi_expansion':EXPANSION,'validation_metrics':result,'selection_score':score},RUN_DIR/'best_model.pth')
pd.DataFrame(history).to_csv(RUN_DIR/'history.csv',index=False); (RUN_DIR/'run_config.json').write_text(json.dumps({'input_size':512,'loss':'ce','roi':'YOLO fixed 1.15 square, external black padding','base_checkpoint':str(BASE),'baseline_metrics':baseline},indent=2)); print('Best:',RUN_DIR/'best_model.pth')


## Grad-CAM display audit

This displays five deterministic validation examples per true grade and saves their figure panels under the run directory. Use the same selected checkpoint for metric and CAM review.

In [ ]:
class GradCAM:
    def __init__(self,model): self.model=model; self.activation=None; self.handle=model.gradcam_target_layer.register_forward_hook(lambda m,i,o:setattr(self,'activation',o))
    def __call__(self,x):
        self.model.eval(); self.model.zero_grad(set_to_none=True); logits=self.model(x.requires_grad_(True)); grade=int(logits.argmax(1)); grad=torch.autograd.grad(logits[0,grade],self.activation)[0]; cam=F.relu((grad.mean((2,3),keepdim=True)*self.activation).sum(1,keepdim=True)); cam=F.interpolate(cam,size=(INPUT_SIZE,INPUT_SIZE),mode='bilinear',align_corners=False)[0,0].detach().cpu().numpy(); return grade,cam/max(float(cam.max()),1e-8)
    def close(self): self.handle.remove()

candidate=Model().to(DEVICE); candidate.load_state_dict(torch.load(RUN_DIR/'best_model.pth',map_location=DEVICE,weights_only=False)['model_state_dict']); cammer=GradCAM(candidate); examples=val_frame.groupby('grade',group_keys=False).head(5).reset_index(drop=True); out=RUN_DIR/'gradcam_examples'; out.mkdir()
for grade in range(5):
    subset=examples[examples.grade==grade]; fig,axes=plt.subplots(len(subset),2,figsize=(8,4*len(subset)))
    for axrow,row in zip(np.atleast_2d(axes),subset.itertuples()):
        image=cv2.imread(row.full_image); roi=square_roi(image,[row.x1,row.y1,row.x2,row.y2]); rgb=cv2.cvtColor(roi,cv2.COLOR_BGR2RGB); processed=np.asarray(transforms.Compose([CLAHE(),transforms.ToPILImage(),transforms.Resize((INPUT_SIZE,INPUT_SIZE))])(rgb)); tensor=val_tf(rgb).unsqueeze(0).to(DEVICE); predicted,cam=cammer(tensor); heat=cv2.cvtColor(cv2.applyColorMap(np.uint8(cam*255),cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB); axrow[0].imshow(processed); axrow[0].set_title(f'True G{grade}'); axrow[1].imshow(cv2.addWeighted(processed,.6,heat,.4,0)); axrow[1].set_title(f'Grad-CAM predicted G{predicted}'); [a.axis('off') for a in axrow]
    plt.tight_layout(); plt.savefig(out/f'grade_{grade}.png',dpi=150); plt.show(); plt.close()
cammer.close(); print('Inspect every displayed panel before considering deployment.')


## Decision

Do not deploy this checkpoint merely because its CAM looks sharper. Require equal-or-better fixed-ROI QWK/macro F1 and inspect all displayed grades for crop-edge attention. If selected, the app must use `IMG_SIZE=512` and `CROP_SIZE=512` with the same YOLO 1.15 square crop.